# 02_feature_engineering: Feature Engineering
Clean the 20 chosen columns, build the 0/1 target, and split the data into train and test (REQ-02).

In [0]:
%run ./00_config

## Step 1: Load the main table

In [0]:
# Read our main table (saved in 01_eda)
loans = spark.table(RAW_TABLE)

print(f"Rows: {loans.count():,}")

## Step 2: Build the 0/1 target
Keep only finished loans. Fully Paid → 0, Charged Off / Default → 1. Pending loans (Current, Late, In Grace Period) are removed because their outcome is unknown.

In [0]:
from pyspark.sql import functions as F

# Keep only finished loans (pending loans have no known outcome)
finished = loans.filter(F.col("loan_status").isin("Fully Paid", "Charged Off", "Default"))

# Target: 1 = default, 0 = repaid
finished = finished.withColumn(
    "label",
    F.when(F.col("loan_status").isin("Charged Off", "Default"), 1).otherwise(0)
)

display(finished.groupBy("label").count())

**658,049 finished loans:** 522,325 repaid (0) and 135,724 defaults (1).

## Step 3: Keep only the 20 features, the label, and the issue date
This also removes all 29 leakage columns. `issue_d` is kept only for the time-based split, not as a model input.

In [0]:
# Keep the 20 features + label + issue_d (for the time split); everything else is dropped
data = finished.select(ALL_FEATURES + ["issue_d", "label"])

# Safety check: no leakage column should be left
leaks_left = [c for c in data.columns if c in LEAKAGE_COLS]

print(f"Columns kept: {len(data.columns)}")
print(f"Leakage columns left: {leaks_left}")

Kept 22 columns (20 features, `issue_d`, `label`). No leakage columns remain.

## Step 4: Test cleaning of term, int_rate, revol_util (keep only digits and dot → number)

In [0]:
# Rule: remove everything except digits and the dot, then convert to a number
test = data.selectExpr(
    "term",       "CAST(regexp_replace(term, '[^0-9.]', '') AS DOUBLE)       AS term_num",
    "int_rate",   "CAST(regexp_replace(int_rate, '[^0-9.]', '') AS DOUBLE)   AS int_rate_num",
    "revol_util", "CAST(regexp_replace(revol_util, '[^0-9.]', '') AS DOUBLE) AS revol_util_num",
)
display(test.limit(5))

The rule works: `" 36 months"` → 36, `"  5.32%"` → 5.32, `"27.9%"` → 27.9.

## Step 5: Test cleaning of emp_length ("< 1 year" → 0, "n/a" → missing + flag)

In [0]:
# emp_length: special cases first, then keep only digits
test = data.selectExpr(
    "emp_length",
    """CASE WHEN emp_length = '< 1 year' THEN 0
            WHEN emp_length = 'n/a'      THEN NULL
            ELSE CAST(regexp_replace(emp_length, '[^0-9]', '') AS DOUBLE)
       END AS emp_length_num""",
    "CASE WHEN emp_length = 'n/a' THEN 1 ELSE 0 END AS emp_length_missing",
)
display(test.groupBy("emp_length", "emp_length_num", "emp_length_missing").count()
            .orderBy("emp_length_num"))

All 12 `emp_length` values map correctly: "< 1 year" → 0, "1 year" → 1 … "10+ years" → 10.
"n/a" (33,672 loans) → missing, with a new flag column `emp_length_missing` = 1.

## Step 6: Test the remaining fixes (inq_last_6mths, dti, annual_inc, home_ownership)

In [0]:
# Remaining fixes from EDA
test = data.selectExpr(
    "CAST(inq_last_6mths AS DOUBLE) AS inq_last_6mths",
    "CASE WHEN dti < 0 OR dti > 100 THEN NULL ELSE dti END AS dti",
    "CASE WHEN annual_inc = 0 THEN NULL ELSE annual_inc END AS annual_inc",
    "CASE WHEN home_ownership IN ('ANY', 'NONE', 'OTHER') THEN 'OTHER' ELSE home_ownership END AS home_ownership",
)

# Check: dti range, how many incomes became missing, home_ownership groups
display(test.selectExpr("MIN(dti)", "MAX(dti)", "SUM(CASE WHEN annual_inc IS NULL THEN 1 ELSE 0 END) AS income_missing"))
display(test.groupBy("home_ownership").count())

`inq_last_6mths` → number; impossible `dti` values → missing (range now 0–98.41);
16 zero incomes → missing; `home_ownership` merged into 4 groups (OTHER = 131).

## Step 7: Pipeline stage 1, cleaning with SQLTransformer
All rules tested in Steps 4–6, in one SQL step. `__THIS__` = the incoming data.

In [0]:
from pyspark.ml.feature import SQLTransformer

# All cleaning rules from Steps 4-6 in one step (cleaned columns get new names)
clean_sql = """
SELECT *,
  CAST(regexp_replace(term, '[^0-9.]', '') AS DOUBLE)       AS term_num,
  CAST(regexp_replace(int_rate, '[^0-9.]', '') AS DOUBLE)   AS int_rate_num,
  CAST(regexp_replace(revol_util, '[^0-9.]', '') AS DOUBLE) AS revol_util_num,
  CASE WHEN emp_length = '< 1 year' THEN 0
       WHEN emp_length = 'n/a'      THEN NULL
       ELSE CAST(regexp_replace(emp_length, '[^0-9]', '') AS DOUBLE) END AS emp_length_num,
  CASE WHEN emp_length IS NULL OR emp_length = 'n/a' THEN 1 ELSE 0 END AS emp_length_missing,
  CAST(inq_last_6mths AS DOUBLE)                             AS inq_last_6mths_num,
  CASE WHEN dti < 0 OR dti > 100 THEN NULL ELSE dti END      AS dti_clean,
  CASE WHEN annual_inc = 0 THEN NULL ELSE annual_inc END     AS annual_inc_clean,
  CASE WHEN home_ownership IN ('ANY', 'NONE', 'OTHER') THEN 'OTHER'
       ELSE home_ownership END                               AS home_ownership_clean
FROM __THIS__
"""
cleaner = SQLTransformer(statement=clean_sql)

# Quick test: run the cleaner on our data and look at the new columns
cleaned = cleaner.transform(data)
display(cleaned.select("term_num", "int_rate_num", "revol_util_num", "emp_length_num",
                       "emp_length_missing", "dti_clean", "home_ownership_clean").limit(5))

The cleaner (SQLTransformer) works: all text columns become clean numbers, and home_ownership has 4 groups.

## Step 8: Pipeline stage 2, fill missing values with the median
Median is robust to extreme values (e.g. an income of $8.9M). It is learned from the training data only.

In [0]:
from pyspark.ml.feature import Imputer

# Columns that still have missing values after cleaning
IMPUTE_COLS = ["mort_acc", "revol_util_num", "dti_clean", "annual_inc_clean",
               "inq_last_6mths_num", "emp_length_num"]

# Fill each one with its median; the filled versions get the suffix "_imp"
imputer = Imputer(
    inputCols=IMPUTE_COLS,
    outputCols=[c + "_imp" for c in IMPUTE_COLS],
    strategy="median",
)
print("Imputer ready for:", IMPUTE_COLS)

Imputer created for 6 columns (median strategy). It will learn the medians from the training data only.

## Step 9: Pipeline stage 3, encode categories (StringIndexer → OneHotEncoder)
Words become number codes, then on/off switches, so the model doesn't treat category codes as amounts.

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

CAT_COLS = ["grade", "home_ownership_clean", "verification_status", "purpose"]

# Move 1: word -> number code ("keep" = unknown words in new data get their own code, no crash)
indexer = StringIndexer(inputCols=CAT_COLS,
                        outputCols=[c + "_idx" for c in CAT_COLS],
                        handleInvalid="keep")

# Move 2: number code -> on/off switches
encoder = OneHotEncoder(inputCols=[c + "_idx" for c in CAT_COLS],
                        outputCols=[c + "_ohe" for c in CAT_COLS])

print("Encoding ready for:", CAT_COLS)

StringIndexer + OneHotEncoder created for 4 category columns. Unknown categories in new data are handled (no crash).

## Step 10: Pipeline stage 4 (VectorAssembler) and the full Pipeline
All model inputs are packed into one `features` column. The 5 stages are chained in order.

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

# Model inputs: clean numbers + filled columns + missing flag + category switches
MODEL_INPUTS = (
    ["loan_amnt", "installment", "delinq_2yrs", "open_acc", "pub_rec",
     "revol_bal", "total_acc", "pub_rec_bankruptcies",      # already clean, no gaps
     "term_num", "int_rate_num"]                             # cleaned, no gaps
    + [c + "_imp" for c in IMPUTE_COLS]                      # cleaned + filled
    + ["emp_length_missing"]                                 # "n/a" flag
    + [c + "_ohe" for c in CAT_COLS]                         # category switches
)

assembler = VectorAssembler(inputCols=MODEL_INPUTS, outputCol="features")

# The full feature Pipeline: 5 stages, in order
feature_pipeline = Pipeline(stages=[cleaner, imputer, indexer, encoder, assembler])

print(f"Model inputs: {len(MODEL_INPUTS)}")
print(f"Pipeline stages: {len(feature_pipeline.getStages())}")

Feature Pipeline: cleaner → imputer → indexer → encoder → assembler (21 inputs → one `features` column).

## Step 11: Finished loans and default rate per issue year (before choosing the split)

In [0]:
# Finished loans per year, and the default rate per year
by_year = (data
           .withColumn("issue_year", F.substring("issue_d", -4, 4).cast("int"))
           .groupBy("issue_year")
           .agg(F.count("*").alias("loans"),
                F.round(100 * F.avg("label"), 2).alias("default_rate_pct"))
           .orderBy("issue_year"))
display(by_year)

2017 has only 11,927 finished loans with a 3.56% default rate: these are mostly early payoffs, not typical loans.
**Split decision:** train on 2012–2015, test on 2016, and exclude 2017.

## Step 12: Time-based split (train 2012–2015, test 2016, 2017 excluded)
The model learns from older loans and is tested on newer ones, as in real use.

In [0]:
# Add the issue year, then split by time (not randomly)
data_y = data.withColumn("issue_year", F.substring("issue_d", -4, 4).cast("int"))

train = data_y.filter(F.col("issue_year") <= 2015)   # the past: model learns here
test  = data_y.filter(F.col("issue_year") == 2016)   # the "future": fair check

for name, df_part in [("Train", train), ("Test", test)]:
    n = df_part.count()
    rate = df_part.agg(F.avg("label")).first()[0] * 100
    print(f"{name}: {n:,} loans, default rate {rate:.2f}%")

Train (2012–2015): 547,821 loans, 20.76% default. Test (2016): 98,301 loans, 21.96% default.
Similar default rates, so the test set is a fair check of future performance.

## Step 13: Fit the Pipeline on train only, then transform train and test

In [0]:
# Learn medians and category lists from TRAIN only
fitted_pipeline = feature_pipeline.fit(train)

# Apply the same learned rules to both sets
train_ready = fitted_pipeline.transform(train)
test_ready  = fitted_pipeline.transform(test)

# Peek: each row now has one "features" vector and its label
display(train_ready.select("features", "label").limit(3))

The Pipeline works: each loan becomes one `features` vector of **45 numbers** (17 numeric inputs + 28 category switches).
Shown in compressed "sparse" form: only non-zero positions are stored.

## Step 14: Save train and test as Delta tables (for 03_training)

In [0]:
# Save both sets so 03_training can read them (overwrite = safe to re-run)
for df_part, table in [(train, TRAIN_TABLE), (test, TEST_TABLE)]:
    (df_part.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table))
    print(f"Saved {spark.table(table).count():,} rows to {table}")

## Step 15: Save the (unfitted) Pipeline recipe for 03_training

In [0]:
# Save the recipe (stages + settings, not the learned values)
feature_pipeline.write().overwrite().save(PIPELINE_PATH)

# Test: load it back and count the stages
loaded = Pipeline.load(PIPELINE_PATH)
print(f"Saved and reloaded: {len(loaded.getStages())} stages")

Pipeline recipe saved to the `artifacts` volume and reloaded successfully (5 stages). `03_training` will load it from there.

# Feature Engineering Summary
- **Target:** Fully Paid → 0, Charged Off / Default → 1. Pending loans (Current, Late, Grace) removed: outcome unknown.
- **Columns:** 20 features + label + issue date. All 29 leakage columns removed.
- **Pipeline (5 stages):** SQLTransformer (clean text numbers, fix impossible values, merge rare categories)
  → Imputer (median) → StringIndexer → OneHotEncoder → VectorAssembler (45 numbers per loan).
- **Split by time:** train 2012–2015 (547,821 loans, 20.76% default), test 2016 (98,301 loans, 21.96% default).
  2017 excluded: too new, mostly early payoffs (3.56% default).
- **Saved:** `train` and `test` Delta tables + the Pipeline recipe (volume `artifacts`).